# Single-Band Sweep + ORx AGC - ADRV9026

Sweep a parameter (TX attenuation or LO2 frequency) and, at **every step**,
auto-level the ORx and capture. The auto-level keeps the captured signal at a
fixed target so each point in the sweep is measured at a comparable level.

## About the ORx "AGC"

There is no trustworthy hardware AGC for the ORx here, so we auto-level in software
on the **captured-IQ peak** (the metric proven faithful on this bench;
`RxDecPowerGet` compresses the range and is not reliable for leveling). At every
sweep point `autolevel_orx` steps the ORx gain index (clamped to the valid
**190..255** window) until the captured peak lands in `ORX_TARGET_DBFS +/- tol`.

It **fails loud**: if the target needs more gain than index 255 (signal too weak)
or less than index 190 (too strong / clipping) it stops and reports a reason rather
than spinning -- watch the `leveled` column and the red X's on the plot.

## 1. Parameters (edit me)

In [ ]:
from adrvtrx import TxChannel, RxChannel

# --- Profile -----------------------------------------------------------------
PROFILES = {
    "98_linksharing":  "ADRV9025Init_StdUseCase98_LinkSharing.profile",   # Tx 491.52 MSPS, Np=12
    "102_linksharing": "ADRV9025Init_StdUseCase102_LinkSharing.profile",  # Np=16
    "14_linksharing":  "ADRV9025Init_StdUseCase14_LinkSharing.profile",
    "50_linksharing":  "ADRV9025Init_StdUseCase50_LinkSharing.profile",
}
PROFILE = "98_linksharing"          # <-- choose here

# --- Signal path (bench wiring TX2->ORx2, TX3->ORx3) -------------------------
TX_CHANNEL  = TxChannel.TX2
ORX_CHANNEL = RxChannel.ORX2
SIGNAL_PATH = "C:/Users/ohammi/OneDrive - aus.edu/DualBand/input_100/Signal1.txt"

# --- Sweep -------------------------------------------------------------------
# SWEEP_KIND: "tx_atten" (dB, 0..41.95) or "lo2_freq" (Hz; LO2 drives TX + ORx).
SWEEP_KIND   = "tx_atten"
SWEEP_VALUES = [20.0, 15.0, 10.0, 5.0, 0.0]      # dB for tx_atten
# Frequency-sweep example:
#   SWEEP_KIND = "lo2_freq"; SWEEP_VALUES = [0.8e9, 0.9e9, 1.0e9, 1.1e9]

# --- ORx auto-level (AGC) ----------------------------------------------------
ORX_TARGET_DBFS  = -20.0   # aim point for the captured peak at every sweep step
ORX_TOLERANCE_DB = 2.0
CAPTURE_MS       = 0.05

## 2. Imports, config, profile

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from adrvtrx.config import load_config
from adrvtrx.profile import read_profile
from adrvtrx.radio import Radio
from adrvtrx.experiment import verify_status
from adrvtrx.waveform import load_tab_iq, full_scale
from adrvtrx.transmit import transmit_bands
from adrvtrx.capture import capture
from adrvtrx.gain import clip_report, autolevel_orx, ORX_GAIN_MIN, ORX_GAIN_MAX
from adrvtrx.sweep import SweepAxis, run_sweep, attenuation_axis, frequency_axis
from adrvtrx._enums import is_orx

cfg = load_config()
cfg.profile_name = PROFILES.get(PROFILE, PROFILE)
info = read_profile(cfg.profile_path)
print("Profile:", cfg.profile_path.name)
print(f"ORx auto-level (AGC) on the captured-IQ peak; valid gain window {ORX_GAIN_MIN}..{ORX_GAIN_MAX}")

## 3. Signal-path summary - sample rates & bit depth

In [ ]:
def _fs(bits):
    return (1 << (bits - 1)) - 1

print(f"{'path':<6}{'rate (MSPS)':>14}{'bits (Np)':>12}{'full scale':>12}")
print(f"{'TX':<6}{info.tx_rate_khz/1000:>14.3f}{info.tx_bits:>12}{_fs(info.tx_bits):>12}")
print(f"{'Rx':<6}{info.rx_rate_khz/1000:>14.3f}{info.rx_bits:>12}{_fs(info.rx_bits):>12}")
print(f"{'ORx':<6}{info.orx_rate_khz/1000:>14.3f}{info.rx_bits:>12}{_fs(info.rx_bits):>12}")

## 4. Load the waveform

In [ ]:
wave = load_tab_iq(SIGNAL_PATH)
print(f"loaded {len(wave)} samples from {SIGNAL_PATH}")

## 5. Connect, program, verify

In [ ]:
radio = Radio(cfg)
radio.connect()
radio.force_safe()
radio.program()
for k, v in verify_status(radio).items():
    print(f"{k}: {v}")

## 6. Build the sweep axis + per-point action (auto-level ORx, capture)

In [ ]:
def measure_peak(ch):
    """Capture `ch` and return (peak_dBFS, ChannelCapture). The trusted level metric."""
    res = capture(radio, int(ch), CAPTURE_MS, bits=info.rx_bits)
    cap = res.channels[ch]
    return clip_report(cap.i, cap.q, info.rx_bits).peak_dbfs, cap

def action(point):
    # AGC: level the ORx on the captured peak before measuring this sweep point.
    lr = autolevel_orx(
        lambda g: radio.set_rx_gain(ORX_CHANNEL, g),
        lambda: measure_peak(ORX_CHANNEL)[0],
        target_dbfs=ORX_TARGET_DBFS, tolerance_db=ORX_TOLERANCE_DB,
    )
    peak, cap = measure_peak(ORX_CHANNEL)           # settled capture
    rep = clip_report(cap.i, cap.q, info.rx_bits)
    rec = dict(point)
    rec.update(orx_gain=lr.final_gain_index, leveled=lr.converged,
               peak_dBFS=peak, railed=rep.railed_samples, note=lr.reason)
    return rec

if SWEEP_KIND == "tx_atten":
    axis = attenuation_axis(radio, TX_CHANNEL, SWEEP_VALUES, name="atten_db")
    XLABEL, XKEY = "TX atten (dB)", "atten_db"
elif SWEEP_KIND == "lo2_freq":
    axis = frequency_axis(radio, "LO2", [int(v) for v in SWEEP_VALUES], name="lo2_hz")
    XLABEL, XKEY = "LO2 (Hz)", "lo2_hz"
else:
    raise ValueError(f"unknown SWEEP_KIND {SWEEP_KIND!r}")

## 7. Run the sweep

In [ ]:
# Start transmitting, then walk the sweep (auto-levels + captures per point).
start_atten = max(SWEEP_VALUES) if SWEEP_KIND == "tx_atten" else 10.0
radio.set_tx_atten(TX_CHANNEL, start_atten)
transmit_bands(radio, {TX_CHANNEL: wave}, info.tx_bits, continuous=True, do_normalize=True)
records = run_sweep([axis], action)
radio.disable_tx()

print(f"{XLABEL:>14} {'orx_gain':>9} {'peak_dBFS':>10} {'leveled':>8} {'railed':>7}  note")
for r in records:
    print(f"{r[XKEY]:>14} {r['orx_gain']:>9} {r['peak_dBFS']:>10.1f} "
          f"{str(r['leveled']):>8} {r['railed']:>7}  {r['note']}")

## 8. Plot - AGC holds the level while gain compensates

In [ ]:
x = np.array([r[XKEY] for r in records], float)
gain = np.array([r["orx_gain"] for r in records])
peak = np.array([r["peak_dBFS"] for r in records])
ok = np.array([r["leveled"] for r in records], bool)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(x, gain, "o-")
ax[0].set_title("settled ORx gain index"); ax[0].set_xlabel(XLABEL)
ax[0].set_ylabel("gain index"); ax[0].grid(True)
ax[1].axhline(ORX_TARGET_DBFS, ls="--", c="k", lw=1, label="target")
ax[1].plot(x[ok], peak[ok], "o-", c="g", label="leveled")
if (~ok).any():
    ax[1].plot(x[~ok], peak[~ok], "x", c="r", ms=9, label="NOT leveled")
ax[1].set_title("captured peak after AGC"); ax[1].set_xlabel(XLABEL)
ax[1].set_ylabel("dBFS"); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()

## 9. Safe-state & disconnect

In [ ]:
radio.safe_state()
radio.disconnect()
print("TX safe, board disconnected")